# Figure 2: OT-tSNE method

Run from the repository root. The last cell writes `figures/method.pdf`.


In [ ]:
from pathlib import Path
import sys

import numpy as np
from IPython.display import Image, display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "src/ne_aligned_utils/methods/ot_graph.py").is_file())
sys.path.insert(0, str(ROOT / "src"))

from ne_aligned_utils.methods import (
    build_ot_tsne_four_block_matrix,
    compute_ot_plan,
    compute_ot_tsne_embedding_from_plan,
    normalize_ot_tsne_graph,
    sparsify_four_block_top_k,
)

seed, n0, n1 = 0, 180, 90


In [ ]:
def create_dataset(
    n0: int = 180, n1: int = 90, seed: int = 0
) -> tuple[np.ndarray, np.ndarray]:
    rng = np.random.RandomState(seed)
    theta0 = np.sort(2 * np.pi * rng.rand(n0))
    theta1 = np.sort(2 * np.pi * rng.rand(n1))
    radius0 = 1.0 + 0.035 * rng.randn(n0)
    radius1 = 1.0 + 0.18 * np.sin(5.0 * theta1 + 0.35) + 0.02 * rng.randn(n1)
    signal0 = np.column_stack(
        (radius0 * np.cos(theta0) - 1.35, radius0 * np.sin(theta0), 0.025 * rng.randn(n0))
    )
    signal1 = np.column_stack(
        (
            radius1 * np.cos(theta1) + 1.35,
            radius1 * np.sin(theta1),
            0.11 * np.sin(3.0 * theta1 + 0.55) + 0.02 * rng.randn(n1),
        )
    )
    return signal0, signal1


x0, x1 = create_dataset(n0=n0, n1=n1, seed=seed)

plan = compute_ot_plan(x0, x1, rel_reg=1.0, tol=1e-12, max_iter=10_000)
embedding, artifacts = compute_ot_tsne_embedding_from_plan(
    plan, effective_k=15, binarize=True, random_state=seed,
    # The figure keeps the schedule it was made with: 200 iterations, the first third
    # exaggerated. Drop both arguments for the benchmark schedule (250 + 500).
    n_epochs=200, early_exaggeration_iter=None, init="random", return_artifacts=True,
)
assert (artifacts["batch0_k"], artifacts["batch1_k"]) == (10, 5)
# The embedding builds its graph block by block and does not keep the dense matrices, so the two
# drawn stages are rebuilt here from the same plan and checked against the graph it used.
affinity = build_ot_tsne_four_block_matrix(plan)
sparse, _split = sparsify_four_block_top_k(affinity, n0=n0, n1=n1, effective_k=15, binarize=True)
assert abs(normalize_ot_tsne_graph(sparse) - artifacts["graph"]).max() < 1e-12


In [ ]:
from ne_aligned_utils.paper.figures.methods import create_figure


In [ ]:
pdf = ROOT / "figures/method.pdf"
png = ROOT / "figures/method.png"
create_figure(x0, x1, plan, affinity, sparse, embedding,
              output_pdf=pdf, output_png=png)
display(Image(filename=str(png), width=1600))
